# Generate Publication Figures — Phase-Aware

This notebook generates all evaluation figures for a given training phase on Google Colab.

## How it works
1. **Setup**: Clones the repo, installs dependencies
2. **Extract hyperparams**: Reads `Train_results/` to find best parameters for the chosen phase
3. **Generate figures**: Runs each `fig*.py` script with phase-appropriate scenario filtering
4. **Collect results**: Shows all generated figures inline, saves logs
5. **Export**: Downloads everything as a ZIP, optionally pushes to GitHub

## Configuration
Set `PHASE` and `BRANCH` in the next cell, then **Run All Cells**.

In [ ]:
# ============================================================
#  CONFIGURATION — edit these values
# ============================================================

# Which phase to generate figures for:
#   1 = Isolated scenarios (KH, Vortex, Tearing, Coalescence)
#   2 = Complex scenarios (Orszag-Tang, MHD Rotor)
#   3 = All 6 scenarios
PHASE = 1

FIGURES = [9]

# Lambda cost for hyperparameter selection
LAMBDA_COST = 0.40

# Git branch to clone from
BRANCH = "main"

# GitHub repo URL
REPO_URL = "https://github.com/armandld/BA_Proj.git"

# GitHub token (needed to push results back)
# Option 1: paste directly
GITHUB_TOKEN = ""

# Option 2: use Colab Secrets (recommended)
try:
    from google.colab import userdata
    GITHUB_TOKEN = userdata.get('GITHUB_TOKEN')
    print("Token loaded from Colab Secrets")
except Exception:
    if GITHUB_TOKEN:
        print("Using manually set token")
    else:
        print("No token set. Push to GitHub will be skipped.")

print(f"Phase: {PHASE}")
print(f"Lambda: {LAMBDA_COST}")
print(f"Branch: {BRANCH}")

In [ ]:
# ============================================================
#  STEP 1: Clone repo & install dependencies
# ============================================================
import os

WORK_DIR = "/content/BA_Proj"

if not os.path.exists(WORK_DIR):
    auth_url = REPO_URL.replace("https://", f"https://{GITHUB_TOKEN}@") if GITHUB_TOKEN else REPO_URL
    !git clone -b {BRANCH} --depth 1 {auth_url} {WORK_DIR}
else:
    # Update existing clone
    %cd {WORK_DIR}
    !git fetch origin {BRANCH}
    !git reset --hard origin/{BRANCH}

%cd {WORK_DIR}
print(f"\nRepo ready at {WORK_DIR}")

In [ ]:
# ============================================================
#  STEP 2: Install Python dependencies from environment.yaml
# ============================================================
!pip install --upgrade pip pyyaml -q

import yaml, subprocess, sys, os

with open("/content/BA_Proj/environment.yaml") as f:
    env = yaml.safe_load(f)

conda_pkgs = []
pip_pkgs = []
for dep in env.get("dependencies", []):
    if isinstance(dep, str) and dep not in ("pip", "python") and not dep.startswith("python="):
        conda_pkgs.append(dep.split("=")[0])
    elif isinstance(dep, dict) and "pip" in dep:
        pip_pkgs.extend(dep["pip"])

all_pkgs = conda_pkgs + pip_pkgs
print(f"Installing {len(all_pkgs)} packages from environment.yaml ...")
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q"] + all_pkgs)

# Verify project imports work
import sys as sys_module
sys_module.path.insert(0, os.path.join(WORK_DIR, 'src'))
try:
    from fig_utils import apply_style, TRAINED_PARAMS
    print(f"\nProject imports OK")
    print(f"Trained threshold_amr: {TRAINED_PARAMS.get('threshold_amr', 'N/A')}")
except Exception as e:
    print(f"WARNING: Project import failed: {e}")

In [ ]:
# ============================================================
#  STEP 3: Generate all figures
# ============================================================
# This runs generate_figures_colab.sh which:
#   - Extracts best hyperparams for the phase
#   - Runs each fig*.py script with individual logging
#   - Saves figures to figures/phase<N>/
#   - Creates a summary JSON

import time

print(f"Generating figures for Phase {PHASE} (lambda={LAMBDA_COST})...")
print("This will take a while. Each figure script prints progress below.\n")

t0 = time.time()

fig_args = f"--figures {' '.join(map(str, FIGURES))}" if FIGURES else ""
!chmod +x generate_figures_colab.sh
!bash generate_figures_colab.sh --phase {PHASE} --lambda {LAMBDA_COST} {fig_args}

elapsed = time.time() - t0
print(f"\nTotal time: {elapsed/60:.1f} minutes ({elapsed/3600:.1f} hours)")

In [ ]:
# ============================================================
#  STEP 4: Display generation summary
# ============================================================
import json

summary_path = f"figures/phase{PHASE}/generation_summary.json"
if os.path.exists(summary_path):
    with open(summary_path) as f:
        summary = json.load(f)
    
    print(f"Phase {summary['phase']} | Lambda {summary['lambda']}")
    print(f"Succeeded: {summary['succeeded']} | Failed: {summary['failed']}")
    print(f"{'='*60}")
    for s in summary['scripts']:
        status_icon = 'OK' if s['status'] == 'ok' else 'FAIL'
        duration = f"{s['duration_s']//60}m{s['duration_s']%60}s"
        print(f"  [{status_icon:4s}] {s['script']:45s} {duration}")
else:
    print("No summary file found. Check if generate_figures_colab.sh ran.")

In [ ]:
# ============================================================
#  STEP 5: Display generated figures
# ============================================================
from IPython.display import display, Image
import glob

fig_dir = f"figures/phase{PHASE}"
pngs = sorted(glob.glob(f"{fig_dir}/*.png"))

print(f"Found {len(pngs)} figures in {fig_dir}/\n")

for p in pngs:
    name = os.path.basename(p)
    print(f"{'='*60}")
    print(f"  {name}")
    print(f"{'='*60}")
    display(Image(filename=p, width=900))
    print()

In [ ]:
# ============================================================
#  STEP 6: Show logs (check for errors/warnings)
# ============================================================
log_dir = f"figures/phase{PHASE}/logs"
log_files = sorted(glob.glob(f"{log_dir}/*.log"))

print(f"Logs for {len(log_files)} scripts:\n")

for lf in log_files:
    name = os.path.basename(lf)
    with open(lf) as f:
        content = f.read()
    lines = content.strip().split('\n')
    
    # Detect status from log content
    has_saved = any('Saved' in l or 'saved' in l for l in lines[-10:])
    has_error = any('Error' in l or 'Traceback' in l for l in lines)
    
    if has_error:
        status = 'ERROR'
    elif has_saved:
        status = 'OK'
    else:
        status = '???'
    
    print(f"[{status:5s}] {name}")
    # Show last 5 lines
    for l in lines[-5:]:
        print(f"    {l}")
    if has_error:
        # Also show the error
        for i, l in enumerate(lines):
            if 'Traceback' in l:
                for el in lines[i:min(i+10, len(lines))]:
                    print(f"    ! {el}")
                break
    print()

In [ ]:
# ============================================================
#  STEP 7: Download results as ZIP
# ============================================================
import shutil

zip_name = f"figures_phase{PHASE}"
zip_path = shutil.make_archive(
    f"/content/{zip_name}",
    'zip',
    root_dir=WORK_DIR,
    base_dir=f"figures/phase{PHASE}"
)

print(f"ZIP created: {zip_path}")
print(f"Size: {os.path.getsize(zip_path) / 1024 / 1024:.1f} MB")

# Auto-download in Colab
try:
    from google.colab import files
    files.download(zip_path)
    print("\nDownload started! Check your browser downloads.")
except ImportError:
    print(f"\nNot on Colab. ZIP saved at: {zip_path}")

In [ ]:
# ============================================================
#  STEP 8 (optional): Push figures to GitHub
# ============================================================
# Only run this cell if you want to push the generated figures
# back to the repository.

PUSH_TO_GITHUB = False  # <-- set to True to enable

if PUSH_TO_GITHUB and GITHUB_TOKEN:
    # Configure git
    !git config user.email "colab-worker@users.noreply.github.com"
    !git config user.name "Colab Figure Generator"
    
    # Set remote with auth
    auth_url = REPO_URL.replace("https://", f"https://{GITHUB_TOKEN}@")
    !git remote set-url origin {auth_url}
    
    # Stage only figures
    !git add figures/phase{PHASE}/
    
    # Show what will be committed
    !echo "\nFiles to commit:"
    !git diff --cached --stat
    
    import subprocess
    n_files = int(subprocess.getoutput("git diff --cached --numstat | wc -l"))
    if n_files > 0:
        !git commit -m "Add Phase {PHASE} figures from Colab ({n_files} files)"
        !git push origin {BRANCH}
        print(f"\nPushed {n_files} files to {BRANCH}")
    else:
        print("\nNo new files to commit.")
elif not GITHUB_TOKEN:
    print("No GITHUB_TOKEN set. Set it in the Configuration cell to enable push.")
else:
    print("Push disabled. Set PUSH_TO_GITHUB = True to enable.")